# 🦥 Fine-Tune Qwen 2.5 3B (Abliterated) on Subtitle Dialogue using Unsloth

**Goal:** Fine-tune `richardyoung/Qwen2.5-3B-Instruct-heretic` on timestamped subtitle transcripts to master:
- Movie-grade screenplay dialogue writing
- Zero-refusal handling of adult, uncensored, and dramatic speech
- Character-consistent voice and style
- Removing disfluencies while preserving dramatic cadence
- Scenario completion and dialogue continuation

**Hardware Target:** Google Colab Free Tier (**Tesla T4 GPU - 15GB VRAM**)
**Export Target:** GGUF `Q4_K_M` optimized for local inference on **GTX 1050 Ti (4GB VRAM)**


### 1. Install Unsloth and Dependencies
Run this cell to install the latest high-efficiency Unsloth kernels.


In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install trl transformers datasets accelerate bitsandbytes


### 2. Upload Your Dataset
Upload the `subtitles_train.jsonl` file generated by your local `scripts/srt_to_jsonl.py` script.


In [ ]:
import os
from google.colab import files

dataset_filename = "subtitles_train.jsonl"

if not os.path.exists(dataset_filename):
    print("📤 Please upload your 'subtitles_train.jsonl' file:")
    uploaded = files.upload()
else:
    print(f"✅ Found existing '{dataset_filename}'")


### 3. Dataset Health Dashboard
Inspect the dataset volume, token counts, and character statistics before allocating GPU memory.


In [ ]:
import json
from collections import Counter

with open(dataset_filename, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

total_rows = len(rows)
char_counter = Counter()
sources = Counter()
est_tokens = []

for row in rows:
    msgs = row.get("messages", [])
    full_text = " ".join(m.get("content", "") for m in msgs)
    tokens = max(1, int(len(full_text) / 3.8))
    est_tokens.append(tokens)
    
    src = row.get("id", "").split("/")[0]
    sources[src] += 1
    
    if len(msgs) >= 3:
        for l in msgs[-1].get("content", "").split("\n"):
            if ":" in l:
                char_counter[l.split(":")[0].strip()] += 1

avg_tokens = sum(est_tokens) / max(1, total_rows)
max_tokens = max(est_tokens) if est_tokens else 0
over_2048 = sum(1 for t in est_tokens if t > 2048)

if total_rows >= 1000:
    verdict = "🟢 GO — Robust dataset ready for training!"
elif total_rows >= 100:
    verdict = "🟡 WARN — Usable dataset size (>=100 rows). Proceeding."
else:
    verdict = "🔴 STOP — Dataset too small (<100 rows). Please add more .srt files."

print("=" * 60)
print("📊 DATASET HEALTH & READINESS REPORT")
print("=" * 60)
print(f"  Total Training Chunks:    {total_rows}")
print(f"  Source Subtitle Files:    {len(sources)}")
print(f"  Unique Characters:        {len(char_counter)}")
print(f"  Avg Estimated Tokens:     ~{avg_tokens:.0f}")
print(f"  Max Estimated Tokens:     ~{max_tokens}")
print(f"  Chunks > 2048 Limit:      {over_2048}")
print("-" * 60)
if char_counter:
    print("🎭 Top Characters:")
    for c, cnt in char_counter.most_common(6):
        print(f"    • {c:20} : {cnt} turns")
    print("-" * 60)
print(f"VERDICT: {verdict}")
print("=" * 60)

if total_rows < 10:
    raise ValueError("Dataset is too small to begin training. Please upload a larger dataset.")


### 4. Load Qwen 2.5 3B with 4-Bit Quantization
Configured specifically for the Tesla T4 free tier.


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Supports 6-line context chunks with headroom
lora_rank = 32         # Balanced rank for learning dialogue nuances without OOM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="richardyoung/Qwen2.5-3B-Instruct-heretic",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # Auto detects float16 for Tesla T4
)


### 5. Setup Fast LoRA Adapters
We target all linear projections (`q, k, v, o` attention projections + `gate, up, down` MLP projections).


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    lora_dropout=0,  # Unsloth is optimized for lora_dropout = 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves ~30% VRAM
    random_state=3407,
)


### 6. Format Dataset with Qwen ChatML Template
We standardize the messages and apply Qwen 2.5's native ChatML chat template.


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

dataset = load_dataset("json", data_files=dataset_filename, split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Sample Formatted Text:\n{dataset[0]['text'][:350]}...")


### 7. Configure SFTTrainer (Optimized for Free T4)
- **Packing:** Groups short subtitle chunks into batches of 2048 tokens, speeding up training ~2-3x.
- **Optimizer:** 8-bit AdamW for low VRAM usage.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,  # Pack shorter chunks together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)


### 8. Execute Fine-Tuning 🚀
Run the training loop and monitor VRAM efficiency.


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name} | Total Memory: {round(gpu_stats.total_memory / (1024**3), 2)} GB")

trainer_stats = trainer.train()

used_vram = round(torch.cuda.max_memory_reserved() / (1024**3), 2)
print(f"🎉 Training finished in {trainer_stats.metrics['train_runtime']:.2f} seconds!")
print(f"Peak VRAM used: {used_vram} GB / {round(gpu_stats.total_memory / (1024**3), 2)} GB")


### 9. Test Fine-Tuned Model Inference
Test if the model generates polished screenplay dialogue with character attribution.


In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": (
            "You are a cinematic script writer. Given raw subtitle dialogue with "
            "timestamps and character tags from a video scene, rewrite it as clean, "
            "natural, movie-grade script dialogue."
        ),
    },
    {
        "role": "user",
        "content": (
            "[Source: test_scene]\n"
            "[SPEAKER_A] [00:00:10 --> 00:00:13] so um what are we doing here exactly\n"
            "[SPEAKER_B] [00:00:13 --> 00:00:16] we're checking the the system perimeter\n"
            "[SPEAKER_A] [00:00:16 --> 00:00:19] did you hear that noise right now\n"
            "[SPEAKER_B] [00:00:19 --> 00:00:22] yeah stay down keep your flashlight off"
        ),
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    repetition_penalty=1.2,
)

print("\n🎬 GENERATED SCRIPT DIALOGUE:\n")
print(tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True))


### 10. Export to GGUF (Optimized for GTX 1050 Ti - 4GB)
Export directly as `Q4_K_M` quantization (~2.1 GB) so it runs with zero VRAM pressure on your local GPU.


In [ ]:
# Save LoRA weights
lora_dir = "qwen25-3b-subtitles-lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"✅ LoRA weights saved to '{lora_dir}'")

# Export merged Q4_K_M GGUF
gguf_dir = "qwen25-3b-subtitles-gguf"
print("📦 Quantizing and converting to GGUF (Q4_K_M)...")
model.save_pretrained_gguf(
    gguf_dir,
    tokenizer,
    quantization_method="q4_k_m",
)

# Download the file to your computer
import glob
gguf_files = glob.glob(f"{gguf_dir}/*.gguf")
if gguf_files:
    print(f"⬇️ Downloading '{gguf_files[0]}' to your local machine...")
    files.download(gguf_files[0])
else:
    print("⚠️ GGUF file generated. Check the folder in the file browser.")
